In [ ]:
!pip -q install gemmi py3Dmol
print("설치 완료 — 이번엔 vina/meeko 없이 가볍게 갑니다")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.7 MB/s eta 0:00:00
설치 완료 — 이번엔 vina/meeko 없이 가볍게 갑니다


In [ ]:
import urllib.request, os
import numpy as np
from scipy import ndimage
import gemmi, py3Dmol

# 단백질이 아닌데 '리간드'로 오해하면 안 되는 것들(물·이온·완충제·당 등)
_SKIP = {"HOH","WAT","NA","CL","CA","MG","ZN","K","MN","FE","CU","SO4","PO4",
         "GOL","EDO","ACT","DMS","PEG","BME","MPD","TRS","EPE","FLC","IMD","NAG","BMA","MAN"}

# ---------- (1) 구조 불러오기 + 리간드 삭제(정답 숨기기) ----------
def load_and_clean(pdb_id):
    """PDB 다운로드 → 공결정 리간드(가장 큰 비표준 residue = 정답 site) 감지
       → 리간드가 있는 chain만 남기고 물·리간드·이온 제거한 '단백질만' 구조."""
    raw, clean = f"{pdb_id}.pdb", f"{pdb_id}_clean.pdb"
    if not os.path.exists(raw):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", raw)
    st = gemmi.read_structure(raw); st.setup_entities()

    # 가장 큰 비표준 residue = 공결정 리간드(정답). 그 중심이 '실제 결합 부위'.
    inst=[]
    for ch in st[0]:
        for res in ch:
            if res.name in _SKIP: continue
            info = gemmi.find_tabulated_residue(res.name)
            if info and (info.is_amino_acid() or info.is_nucleic_acid()): continue
            inst.append((ch.name, res.name, res.seqid.num,
                         [[a.pos.x,a.pos.y,a.pos.z] for a in res]))
    lchain,lname,lseq,coords = max(inst, key=lambda t: len(t[3]))
    true_site = np.array(coords).mean(0).round(2).tolist()

    # 정답 리간드의 PDB 블록만 따로 보관(마지막에 '정답 공개'할 때 겹쳐 그림)
    lig_lines=[l for l in open(raw) if l.startswith(("HETATM","ATOM"))
               and l[17:20].strip()==lname and l[21]==lchain and l[22:26].strip()==str(lseq)]
    lig_block="".join(lig_lines)+"END\n"

    # 단백질만 정리 → 리간드가 있던 chain만 남김(이량체에서 반대쪽 주머니와 헷갈리지 않게)
    st.remove_alternative_conformations(); st.remove_hydrogens()
    st.remove_ligands_and_waters(); st.remove_empty_chains(); st.write_pdb("_tmp.pdb")
    with open(clean,"w") as f:
        for l in open("_tmp.pdb"):
            if l.startswith(("ATOM","HETATM","ANISOU","TER")) and l[21]!=lchain: continue
            f.write(l)

    print(f"[{pdb_id}] 불러오기 완료 · 공결정 리간드 {lname}({lchain}{lseq}) · chain {lchain}만 사용 · (정답 site는 나중에 공개)")
    return {"pdb":pdb_id, "raw":raw, "clean":clean, "lig":lig_block,
            "lig_name":lname, "lig_chain":lchain, "true_site":true_site}

# ---------- (2) binding site 예측 (LIGSITE — 단백질 모양만 사용) ----------
def predict_site(clean_pdb, step=1.2):
    """격자를 깔고, 각 빈 격자점이 '여러 방향에서 단백질로 둘러싸였나'를 센다.
       사방이 막힌(=파묻힌) 점들의 가장 큰 덩어리 = 예측 결합 부위. 리간드 정보는 전혀 안 씀."""
    A=[l for l in open(clean_pdb) if l.startswith("ATOM")
       and (l[76:78].strip() or l[12:16].strip()[:1])!="H"]
    A=np.array([[float(x[30:38]),float(x[38:46]),float(x[46:54])] for x in A])
    lo=A.min(0)-5; hi=A.max(0)+5; dims=tuple(np.ceil((hi-lo)/step).astype(int))
    occ=np.zeros(dims,bool); ai=np.floor((A-lo)/step).astype(int); rad=int(round(2.7/step))
    for dx in range(-rad,rad+1):                    # 원자를 반경 ~2.7 A 공으로 채워 '단백질이 찬 곳' 표시
      for dy in range(-rad,rad+1):
        for dz in range(-rad,rad+1):
          if dx*dx+dy*dy+dz*dz<=rad*rad:
            c=ai+[dx,dy,dz]; mm=(c>=0).all(1)&(c<np.array(dims)).all(1); cc=c[mm]
            occ[cc[:,0],cc[:,1],cc[:,2]]=True
    def enc(ax,ms):                                 # 축 ax 방향으로 양쪽이 단백질에 막혔나?
        a=np.zeros(occ.shape,bool); b=np.zeros(occ.shape,bool)
        for s in range(1,ms+1): a|=np.roll(occ,-s,ax); b|=np.roll(occ,s,ax)
        return a&b
    ms=int(round(9/step)); score=sum(enc(a,ms).astype(int) for a in range(3))
    pocket=(~occ)&(score==3)                         # 빈 공간이면서 3축 모두 둘러싸임 = 파묻힌 곳
    if pocket.sum()<10: pocket=(~occ)&(score>=2)     # 너무 적으면 기준 완화(2축)
    lab,n=ndimage.label(pocket)                      # 파묻힌 점들을 덩어리로 묶고
    sz=ndimage.sum(np.ones_like(lab),lab,range(1,n+1))
    idx=np.argwhere(lab==(np.argmax(sz)+1))          # 가장 큰 덩어리 = 예측 결합 부위
    return (lo+idx.mean(0)*step).round(2).tolist()

# ---------- (3) 눈으로 확인하는 함수들 ----------
def show_cleaning(m):
    """왼쪽=원본(리간드 빨강) / 오른쪽=정리 후(단백질만). 뭘 지웠는지 대조."""
    v=py3Dmol.view(width=880,height=440,viewergrid=(1,2))
    v.addModel(open(m["raw"]).read(),"pdb",viewer=(0,0))
    v.setStyle({"cartoon":{"color":"lightgray"}},viewer=(0,0))
    v.addStyle({"hetflag":True},{"stick":{"colorscheme":"redCarbon"},"sphere":{"scale":0.3}},viewer=(0,0))
    v.addModel(open(m["clean"]).read(),"pdb",viewer=(0,1))
    v.setStyle({"cartoon":{"color":"lightgray"}},viewer=(0,1))
    v.zoomTo(); v.setBackgroundColor("white")
    print("왼쪽 = 원본 (빨강 = 리간드·물·이온)   →   오른쪽 = 정리 후 (단백질만)")
    print("→ 리간드까지 다 지웠으니, 이제 '어디가 결합 부위인지' 모른다. 예측할 차례.")
    v.show()

def show_protein(m, site=None, box=10, title=""):
    """단백질(회색) + (선택) 예측한 자리 빨강 박스."""
    v=py3Dmol.view(width=560,height=460)
    v.addModel(open(m["clean"]).read(),"pdb"); v.setStyle({"cartoon":{"color":"lightgray"}})
    if site is not None:
        v.addBox({"center":{"x":site[0],"y":site[1],"z":site[2]},
                  "dimensions":{"w":box,"h":box,"d":box},"color":"red","opacity":0.6,"wireframe":True})
    v.zoomTo(); v.setBackgroundColor("white")
    if title: print(title)
    v.show()

def reveal_answer(m, site, box=10):
    """숨겨뒀던 실제 리간드(주황)를 꺼내 예측 박스(빨강)와 겹쳐보고, 두 중심 거리로 채점."""
    d=float(np.linalg.norm(np.array(site)-np.array(m["true_site"])))
    print(f"예측한 site 중심 : {site}")
    print(f"실제 리간드 중심 : {m['true_site']}   (숨겨뒀던 {m['lig_name']})")
    print(f"→ 두 중심 거리 : {d:.2f} A  →  {'OK 명중! 같은 주머니를 찾았다' if d<5 else '빗나감 (다른 주머니)'}")
    v=py3Dmol.view(width=560,height=460)
    v.addModel(open(m["clean"]).read(),"pdb"); v.setStyle({"cartoon":{"color":"lightgray"}})
    v.addModel(m["lig"],"pdb"); v.setStyle({"model":1},{"stick":{"colorscheme":"orangeCarbon"}})
    v.addBox({"center":{"x":site[0],"y":site[1],"z":site[2]},
              "dimensions":{"w":box,"h":box,"d":box},"color":"red","opacity":0.6,"wireframe":True})
    v.zoomTo({"model":1}); v.setBackgroundColor("white")
    print("주황 = 실제 리간드(숨겨뒀던 정답), 빨강 박스 = 단백질 모양만으로 예측한 자리.")
    v.show()
    return d

print("셋업 완료 — load_and_clean · predict_site · show_cleaning · show_protein · reveal_answer 준비됨")

셋업 완료 — load_and_clean · predict_site · show_cleaning · show_protein · reveal_answer 준비됨


In [ ]:
m = load_and_clean("1IEP")   # STI(이매티닙) 자동 감지 → 지우고 단백질만 남김
show_cleaning(m)

[1IEP] 불러오기 완료 · 공결정 리간드 STI(A201) · chain A만 사용 · (정답 site는 나중에 공개)
왼쪽 = 원본 (빨강 = 리간드·물·이온)   →   오른쪽 = 정리 후 (단백질만)
→ 리간드까지 다 지웠으니, 이제 '어디가 결합 부위인지' 모른다. 예측할 차례.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
site = predict_site(m["clean"])
print("예측 binding site 중심:", site)
show_protein(m, site=site, title="빨강 박스 = 단백질 모양만으로 예측한 주머니")

예측 binding site 중심: [17.34, 52.27, 15.11]
빨강 박스 = 단백질 모양만으로 예측한 주머니


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
reveal_answer(m, site)

예측한 site 중심 : [17.34, 52.27, 15.11]
실제 리간드 중심 : [15.61, 53.38, 15.45]   (숨겨뒀던 STI)
→ 두 중심 거리 : 2.08 A  →  OK 명중! 같은 주머니를 찾았다
주황 = 실제 리간드(숨겨뒀던 정답), 빨강 박스 = 단백질 모양만으로 예측한 자리.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

2.0834106652314133

In [ ]:
m2 = load_and_clean("1ERE")        # EST(에스트라디올) 자동 감지 → 단백질만 남김
site2 = predict_site(m2["clean"])
print("예측 binding site 중심:", site2)
show_protein(m2, site=site2, title="ER: 단백질 모양만으로 예측한 주머니")

[1ERE] 불러오기 완료 · 공결정 리간드 EST(A600) · chain A만 사용 · (정답 site는 나중에 공개)
예측 binding site 중심: [8.02, 48.54, 128.39]
ER: 단백질 모양만으로 예측한 주머니


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
reveal_answer(m2, site2)

예측한 site 중심 : [8.02, 48.54, 128.39]
실제 리간드 중심 : [9.27, 46.29, 131.21]   (숨겨뒀던 EST)
→ 두 중심 거리 : 3.82 A  →  OK 명중! 같은 주머니를 찾았다
주황 = 실제 리간드(숨겨뒀던 정답), 빨강 박스 = 단백질 모양만으로 예측한 자리.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3.8180361444072424